---
# Brownian and MSD plots
---

In [ ]:

# paste your scene code here
from manim import *
import numpy as np
config.background_color = WHITE
class TransposedConvolutionVisualization(Scene):
    def construct(self):
        S = 0.5  # speed multiplier — lower = faster

        # Define the input, kernel, and output dimensions
        input_data = np.array([[1, 2], [3, 4]])
        kernel = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]])  # Normalized for visualization
        
        # Calculate expected output (4x4 for 2x2 input with 3x3 kernel)
        output_shape = (4, 4)
        
        # Create input grid (2x2)
        input_grid = self.create_grid(2, 2, cell_size=0.8, position=LEFT * 4)
        input_values = VGroup()
        
        for i in range(2):
            for j in range(2):
                value = Text(str(input_data[i, j]), font_size=24,color = BLACK)
                value.move_to(input_grid[i * 2 + j].get_center())
                input_values.add(value)
        
        input_label = Text("Input (2×2)", font_size=32, color = BLACK).scale(0.6).next_to(input_grid, DOWN)
        
        # Create kernel grid (3x3)
        kernel_grid = self.create_grid(3, 3, cell_size=0.5, position=ORIGIN)
        kernel_values = VGroup()
        
        for i in range(3):
            for j in range(3):
                value = Text(f"{kernel[i, j]}", font_size=16, color = BLACK)
                value.move_to(kernel_grid[i * 3 + j].get_center())
                kernel_values.add(value)
        
        kernel_label = Text("Kernel (3×3)", font_size=32, color = BLACK).scale(0.6).next_to(kernel_grid, DOWN)
        
        # Create output grid (4x4)
        output_grid = self.create_grid(4, 4, cell_size=0.6, position=RIGHT * 4)
        output_values = VGroup()
        output_data = np.zeros((4, 4))
        
        for i in range(4):
            for j in range(4):
                value = Text("0", font_size=16, color=BLACK)
                value.move_to(output_grid[i * 4 + j].get_center())
                output_values.add(value)
        
        output_label = Text("Output (4×4)", font_size=32, color=BLACK).scale(0.6).next_to(output_grid, DOWN)
        
        # Show all grids
        self.play(
            FadeIn(input_grid), FadeIn(input_values), FadeIn(input_label),
            FadeIn(kernel_grid), FadeIn(kernel_values), FadeIn(kernel_label),
            FadeIn(output_grid), FadeIn(output_values), FadeIn(output_label),
            run_time=S
        )
        self.wait(2 * S)
        
        # Explanation text
        explanation = Paragraph(
            "Each input cell is multiplied by the kernel",
            "and accumulated into the corresponding output positions",
            alignment="center",
            font_size=24,color=BLACK
        ).scale(0.7).to_edge(DOWN)
        self.play(Write(explanation), run_time=S)
        self.wait(2 * S)
        
        # Process each input cell
        for input_i in range(2):
            for input_j in range(2):
                self.next_section()

                # Highlight current input cell
                current_input_idx = input_i * 2 + input_j
                current_input_cell = input_grid[current_input_idx]
                current_input_value = input_values[current_input_idx]
                
                self.play(
                    current_input_cell.animate.set_fill(YELLOW, opacity=0.5),
                    current_input_value.animate.set_color(BLACK),
                    run_time=S
                )
                
                # Show multiplication with kernel
                multiplication_text = Text(
                    f"Input[{input_i},{input_j}] = {input_data[input_i, input_j]} × Kernel",
                    font_size=32, color=BLACK
                ).scale(0.6).next_to(kernel_grid, UP)
                self.play(Write(multiplication_text), run_time=S)
                
                # Create and animate the patch
                patch_grid = self.create_grid(3, 3, cell_size=0.5, position=UP * 2.5)
                patch_values = VGroup()
                
                for k_i in range(3):
                    for k_j in range(3):
                        result_value = input_data[input_i, input_j] * kernel[k_i, k_j]
                        value = Text(f"{result_value}", font_size=16, color=RED)
                        value.move_to(patch_grid[k_i * 3 + k_j].get_center())
                        patch_values.add(value)
                
                patch_label = Text("Resulting Patch", font_size=26, color=BLACK).scale(0.6).next_to(patch_grid, UP)
                
                self.play(FadeIn(patch_grid), FadeIn(patch_values), FadeIn(patch_label), run_time=S)
                self.wait(2 * S)
                
                # Move patch to output position and add to accumulator
                output_start_i = input_i
                output_start_j = input_j
                
                # Animate moving each patch value to its output position
                patch_to_output_anims = []
                
                for k_i in range(3):
                    for k_j in range(3):
                        output_i = output_start_i + k_i
                        output_j = output_start_j + k_j
                        
                        if output_i < 4 and output_j < 4:
                            patch_idx = k_i * 3 + k_j
                            output_idx = output_i * 4 + output_j
                            
                            # Update the output data
                            old_value = output_data[output_i, output_j]
                            new_value = old_value + input_data[input_i, input_j] * kernel[k_i, k_j]
                            output_data[output_i, output_j] = new_value
                            
                            # Create animation to move patch value to output
                            target_pos = output_grid[output_idx].get_center() + np.array([0.2, 0.2, 0])  # Offset to top-right corner
                            patch_to_output_anims.append(
                                patch_values[patch_idx].animate.move_to(target_pos).set_font_size(12)  # Reduce font size
                            )
                
                self.play(*patch_to_output_anims, run_time=S)
                self.wait(0.5 * S)
                
                # Update output values
                for i in range(4):
                    for j in range(4):
                        output_idx = i * 4 + j
                        new_text = Text(f"{int(output_data[i, j])}", font_size=16, color=PURPLE if output_data[i, j] > 0 else BLACK)
                        new_text.move_to(output_grid[output_idx].get_center())
                        self.play(Transform(output_values[output_idx], new_text), run_time=0.3 * S)
                
                # Clean up patch
                self.play(
                    FadeOut(patch_grid), 
                    FadeOut(patch_values), 
                    FadeOut(patch_label),
                    FadeOut(multiplication_text),
                    run_time=S
                )
                
                # Reset input cell highlighting
                self.play(
                    current_input_cell.animate.set_fill(BLACK, opacity=0),
                    current_input_value.animate.set_color(BLACK),
                    run_time=S
                )
                
                self.wait(1 * S)
    

    def create_grid(self, rows, cols, cell_size=0.5, position=ORIGIN):
        """Create a grid of squares"""
        grid = VGroup()
        
        for i in range(rows):
            for j in range(cols):
                square = Square(side_length=cell_size)
                square.set_stroke(BLACK, 2)
                square.set_fill(BLACK, opacity=0)
                
                # Position the square
                x_pos = (j - cols/2 + 0.5) * cell_size
                y_pos = (rows/2 - i - 0.5) * cell_size
                square.move_to([x_pos, y_pos, 0] + position)
                
                grid.add(square)
        
        return grid
%manim -ql TransposedConvolutionVisualization


Manim Community v0.18.1

[04/27/26 01:30:00] INFO     Animation 0 : Partial movie file written in                   ]8;id=180989;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=495754;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/home/jeroen/Manim/manim-visualizations/transposed_convoluti                         
                             on/media/videos/transposed_convolution/480p15/partial_movie_f                         
                             iles/TransposedConvolutionVisualization/931832650_2026915578_                         
                             223132457.mp4'                                                                        

                    INFO     Animation 1 : Partial movie file written in                   ]8;id=26872;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=289199;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/home/jeroen/Manim/manim-visualizations/transposed_convoluti                         
                             on/media/videos/transposed_convolution/480p15/partial_movie_f                         
                             iles/TransposedConvolutionVisualization/1905288516_4042641242                         
                             _473483826.mp4'                                                                       

[04/27/26 01:30:02] INFO     Animation 2 : Partial movie file written in                   ]8;id=35493;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=252776;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/home/jeroen/Manim/manim-visualizations/transposed_convoluti                         
                             on/media/videos/transposed_convolution/480p15/partial_movie_f                         
                             iles/TransposedConvolutionVisualization/1905288516_3072469510                         
                             _909688948.mp4'                                                                       

[04/27/26 01:30:03] INFO     Animation 3 : Partial movie file written in                   ]8;id=605003;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=452024;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/home/jeroen/Manim/manim-visualizations/transposed_convoluti                         
                             on/media/videos/transposed_convolution/480p15/partial_movie_f                         
                             iles/TransposedConvolutionVisualization/1905288516_4042641242                         
                             _2499829052.mp4'                                                                      

...

[04/27/26 01:32:06] INFO     Animation 83 : Partial movie file written in                  ]8;id=163520;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=374124;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/home/jeroen/Manim/manim-visualizations/transposed_convoluti                         
                             on/media/videos/transposed_convolution/480p15/partial_movie_f                         
                             iles/TransposedConvolutionVisualization/1905288516_2143829895                         
                             _1433206656.mp4'                                                                      

[04/27/26 01:32:08] INFO     Animation 84 : Partial movie file written in                  ]8;id=57007;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=328342;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/home/jeroen/Manim/manim-visualizations/transposed_convoluti                         
                             on/media/videos/transposed_convolution/480p15/partial_movie_f                         
                             iles/TransposedConvolutionVisualization/1905288516_3832781150                         
                             _3187927348.mp4'                                                                      

[04/27/26 01:32:10] INFO     Animation 85 : Partial movie file written in                  ]8;id=852061;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=895539;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/home/jeroen/Manim/manim-visualizations/transposed_convoluti                         
                             on/media/videos/transposed_convolution/480p15/partial_movie_f                         
                             iles/TransposedConvolutionVisualization/1905288516_3297088275                         
                             _2627877369.mp4'                                                                      

[04/27/26 01:32:12] INFO     Animation 86 : Partial movie file written in                  ]8;id=602687;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=57092;file:///home/jeroen/.venvs/manim/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/home/jeroen/Manim/manim-visualizations/transposed_convoluti                         
                             on/media/videos/transposed_convolution/480p15/partial_movie_f                         
                             iles/TransposedConvolutionVisualization/1905288516_94350373_9                         
                             68638240.mp4'                                                                         

Animation 87: Transform(Text('0')):   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
from manim import *
import numpy as np

config.background_color = WHITE
# https://github.com/3b1b/manim

# Scene is composed as follows:

# 1. Precompute max MSD across all runs (synchronised RNG)
# 2. Build axes and labels
# 3. Persistent step counter in upper right corner
# 4. Loop over different particle counts:
#    a. Simulate trajectories and precompute MSD for animation
#    b. Initialize MSD curve and particle dots/paths
#    c. Updater function to animate particles and MSD curve

# Manim was not made to be commented. It is an evil language library which you simply have to spend hours and survive, and figure it out after. Lots of small steps accumulate into one big scene which makes it super hard to read :)
# Code has examples in the manim documentation, and in 3b1b git. Auto-completed code was used for MSD calculations.  

from manim import *
import numpy as np

config.background_color = WHITE


class RandomWalkAligned(Scene):
    def construct(self):
        speed = 1
        dt_fixed = 0.01
        T = 3.01
        n_steps = int(T / dt_fixed)

        particle_counts = [1, 4, 100]
        base_colors = [RED, BLUE, PURPLE, ORANGE, TEAL, PINK]

        def simulate_walk(n_particles, seed=6):
            np.random.seed(seed)
            traj_all = []

            for _ in range(n_particles):
                pos = np.array([0.0, 0.0])
                traj = []

                for _ in range(n_steps):
                    pos += np.random.normal(0, speed * np.sqrt(dt_fixed), size=2)
                    traj.append(pos.copy())

                traj_all.append(np.array(traj))

            traj_all = np.array(traj_all)
            r0 = traj_all[:, 0, :]
            msd = np.mean(np.sum((traj_all - r0[:, None, :]) ** 2, axis=2), axis=0)

            return traj_all, msd

        all_msd_max = 0
        for n_particles in particle_counts:
            _, msd_tmp = simulate_walk(n_particles)
            all_msd_max = max(all_msd_max, msd_tmp[-1])

        axes = Axes(
            x_range=[-5, 5, 1],
            y_range=[-5, 5, 1],
            x_length=5,
            y_length=5,
            axis_config={"color": BLUE, "tip_length": 0.1, "tip_width": 0.1},
        ).to_edge(LEFT, buff=1)

        x_label = Text("x", color=BLACK, font_size=24).move_to(axes.c2p(-5.5, 0))
        y_label = Text("y", color=BLACK, font_size=24).move_to(axes.c2p(0, 5.5))
        xy_labels = VGroup(x_label, y_label)

        msd_axes = Axes(
            x_range=[0, T, T / 5],
            y_range=[0, all_msd_max * 1.1, (all_msd_max * 1.1) / 5],
            x_length=5,
            y_length=5,
            axis_config={"color": GREEN, "tip_length": 0.1, "tip_width": 0.1},
        ).to_edge(RIGHT, buff=1)

        t_label = Text("t", color=BLACK, font_size=24).move_to(msd_axes.c2p(T / 2, -1))
        msd_label = Text("Mean Squared Displacement", color=BLACK, font_size=24)
        msd_label.move_to(msd_axes.c2p(0, (all_msd_max * 1.1) / 2)).rotate(PI / 2).shift(LEFT * 0.3)
        msd_labels = VGroup(t_label, msd_label)

        self.add(axes, xy_labels, msd_axes, msd_labels)

        step_tracker = ValueTracker(0)
        step_text = Text("Number of steps taken:", font_size=36, color=BLACK)
        step_counter = Integer(0, group_with_commas=True, color=BLACK)
        step_counter.add_updater(lambda m: m.set_value(int(step_tracker.get_value())))
        display = VGroup(step_text, step_counter).arrange(RIGHT, buff=0.2).scale(0.7).to_corner(UR)
        self.add(display)

        msd_curves = []
        times_pre = np.arange(1, n_steps + 1) * dt_fixed

        for run_idx, n_particles in enumerate(particle_counts):
            np.random.seed(6)
            run_color = base_colors[run_idx % len(base_colors)]
            traj_all, msd_pre = simulate_walk(n_particles)

            bottom_label = Text(
                f"Brownian motion with {n_particles} particles",
                font_size=32,
                color=BLACK,
            ).to_edge(DOWN).shift(DOWN * 0.3)

            self.play(Write(bottom_label))

            msd_curve = VMobject(color=run_color).set_stroke(width=4)
            msd_curve.start_new_path(msd_axes.c2p(0, 0))

            for curve in msd_curves:
                curve.set_stroke(opacity=0.2)

            self.add(msd_curve)
            msd_curves.append(msd_curve)

            dots = []
            paths = []

            for p in range(n_particles):
                start = axes.c2p(0, 0)
                opacity = max(0.1, 1 - p / n_particles)

                dot = Dot(start, radius=0.08, color=run_color).set_opacity(opacity)
                path = VMobject().set_points_as_corners([start, start])
                path.set_color(run_color)
                path.set_stroke(width=3, opacity=0.3 * opacity)

                self.add(path, dot)
                dots.append(dot)
                paths.append(path)

            step_index = 0
            accumulated_time = 0

            def update(mob, dt):
                nonlocal step_index, accumulated_time

                accumulated_time += dt

                while accumulated_time >= dt_fixed and step_index < n_steps:
                    accumulated_time -= dt_fixed

                    for p in range(n_particles):
                        pos = traj_all[p, step_index]
                        new_point = axes.c2p(pos[0], pos[1])
                        dots[p].move_to(new_point)
                        paths[p].add_line_to(new_point)

                    msd_curve.add_line_to(msd_axes.c2p(times_pre[step_index], msd_pre[step_index]))
                    step_tracker.increment_value(1)
                    step_index += 1

            self.wait(1)
            dots[0].add_updater(update)
            self.wait(T)
            dots[0].remove_updater(update)

            self.wait(1)
            self.play(FadeOut(bottom_label))

            keep = [axes, xy_labels, msd_axes, msd_labels, display, *msd_curves]
            self.remove(*[m for m in self.mobjects if m not in keep])

            step_tracker.set_value(0)
%manim -ql RandomWalkAligned
